In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.chdir('..')

In [ ]:
import torch
from src.config.vit_config import ViTConfig
from src.data.transformers_dataset import MelSpectrogramDataset
from src.training.vit_trainer import ViTTrainer
from src.utils.metrics import compute_metrics
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    TrainingArguments,
    EarlyStoppingCallback,
)

In [ ]:
device = ViTConfig.get_device()
image_processor = ViTImageProcessor.from_pretrained(ViTConfig.MODEL_ID)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [ ]:
model = ViTForImageClassification.from_pretrained(
    ViTConfig.MODEL_ID,
    num_labels              = ViTConfig.NUM_CLASSES,
    id2label                = ViTConfig.ID2LABEL,
    label2id                = ViTConfig.LABEL2ID,
    ignore_mismatched_sizes = True,
)
print(f"\n  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



  Trainable parameters: 85,800,194


In [8]:
print(f"  GPUs available: {torch.cuda.device_count()}")
model.to(device)

  GPUs available: 1


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
train_ds = MelSpectrogramDataset(ViTConfig.TRAIN_PT, image_processor)
val_ds   = MelSpectrogramDataset(ViTConfig.VAL_PT,   image_processor)

  Loaded /kaggle/input/datasets/trieuvuongnguyen/for-preprocessed/processed/train.pt: 53866 samples
  Loaded /kaggle/input/datasets/trieuvuongnguyen/for-preprocessed/processed/val.pt: 10798 samples


In [ ]:
training_args = TrainingArguments(
    output_dir                  = ViTConfig.OUTPUT_DIR,
    num_train_epochs            = ViTConfig.EPOCHS,
    per_device_train_batch_size = ViTConfig.BATCH_SIZE,
    per_device_eval_batch_size  = ViTConfig.BATCH_SIZE * 2,
 
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "val_acc",
    greater_is_better           = True,
    save_total_limit            = 2,
 
    logging_dir                 = f"{ViTConfig.OUTPUT_DIR}/logs",
    logging_strategy            = "steps",
    logging_steps               = 50,
    report_to                   = "none",
 
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    seed                        = ViTConfig.SEED,
    label_names                 = ["labels"],
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
trainer = ViTTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=5)],
)

In [12]:
print("\n  Starting ViT fine-tuning …\n")
trainer.train()


  Starting ViT fine-tuning …



Epoch,Training Loss,Validation Loss,Val Acc
1,0.019587,0.004902,0.999167
2,0.029827,0.000766,0.999815
3,0.014866,0.004148,0.998889
4,0.011309,0.046306,0.993425
5,0.000037,0.004496,0.999259
6,0.000063,0.005382,0.998889
7,0.000024,0.001394,0.999722


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=23569, training_loss=0.00967739891682845, metrics={'train_runtime': 4492.7534, 'train_samples_per_second': 239.791, 'train_steps_per_second': 14.989, 'total_flos': 2.921928458805729e+19, 'train_loss': 0.00967739891682845, 'epoch': 7.0})

In [ ]:
trainer.save_model(ViTConfig.FINAL_DIR)
image_processor.save_pretrained(ViTConfig.FINAL_DIR)
print(f"\n  Model saved {ViTConfig.FINAL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Model saved → ./vit-fakeaudio-final


In [14]:
print("\n  Final evaluation:")
print(trainer.evaluate())


  Final evaluation:


{'eval_loss': 0.0007663057767786086, 'eval_val_acc': 0.9998147805149101, 'eval_runtime': 35.8092, 'eval_samples_per_second': 301.542, 'eval_steps_per_second': 9.439, 'epoch': 7.0, 'train_acc': 0.0}
